# 1.5 · 窗口函数 / Window Functions ⭐

> **课程定位 / Where this fits**
> **Part 1 第 5 课**——SQL 最重要、最被低估的特性。学完之后你会觉得**没有窗口函数的 SQL 是残废 SQL**。
> **Part 1, lesson 5** — the single most important SQL feature you may not have learned. After this, SQL without window functions feels crippled.

> 📐 **符号约定 / Notation**
> 窗口 / window：对每一行"可见"的一段邻居行集合，由 `OVER (...)` 决定。
> Window = the set of neighbour rows visible to one source row, set by `OVER (...)`.

> 💡 **面试相关 / Interview-relevant**
> - "每组 top-3" / "每组第 N 名" ★★★★★（**必考**）
> - "用户次留 / 7 留" ★★★★★
> - "**连续**登录 7 天的用户" ★★★★★（gaps-and-islands）
> - "运行累计 / 7 日移动平均" ★★★★
> - "**`RANK` vs `DENSE_RANK` vs `ROW_NUMBER`** 的差别" ★★★★
> - "怎么求每个用户的**首次/末次**购买" ★★★★

---

## 学习目标 / Learning Objectives

学完本节，你应该能：
After this notebook you'll be able to:

1. 解释窗口函数 vs `GROUP BY` 的**根本差别**：窗口函数**不塌缩行**。
   Explain windows vs `GROUP BY`: windows preserve row count.
2. 读懂任意 `OVER (PARTITION BY ... ORDER BY ... ROWS BETWEEN ...)` 子句。
   Parse any `OVER (PARTITION BY ... ORDER BY ... ROWS BETWEEN ...)` clause.
3. 写出 8 个核心窗口函数：`ROW_NUMBER`, `RANK`, `DENSE_RANK`, `NTILE`, `LAG`, `LEAD`, `FIRST_VALUE`, `LAST_VALUE`。
   Write the 8 core window functions.
4. 实现**累计聚合**和**滑动窗口**（含 7 日移动均线）。
   Implement running totals and rolling windows (incl. 7-day MA).
5. 用窗口函数解决 **top-N per group**、**Δ vs 上一行**、**连续 N 天**等高频面试题。
   Solve top-N per group, period-over-period delta, consecutive-N-day patterns.

---

## 目录 / Table of Contents

1. [窗口函数是什么 / What Is a Window Function](#1)
2. [`OVER` 子句的三大要素 / The Three Parts of `OVER`](#2)
3. [排名函数 / Ranking: `ROW_NUMBER`, `RANK`, `DENSE_RANK`, `NTILE`](#3)
4. [偏移函数 / Offset: `LAG`, `LEAD`, `FIRST_VALUE`, `LAST_VALUE`](#4)
5. [聚合作为窗口 / Aggregates as Windows](#5)
6. [⭐ 帧子句 / The Frame Clause `ROWS BETWEEN ...`](#6)
7. [累计聚合 / Running Totals](#7)
8. [滑动窗口 / Rolling Windows + 7 日移动均线](#8)
9. [`PERCENT_RANK` & `CUME_DIST`](#9)
10. [⭐ 实战：top-N per group + 同期对比 + 连续 N 天 / Hands-on](#10)
11. [小结 / Summary](#11)


<a id="1"></a>
## 1. 窗口函数是什么 / What Is a Window Function

### 1.1 vs GROUP BY 的根本差别

| 维度 / Aspect | `GROUP BY` 聚合 | 窗口函数 |
|---|---|---|
| 输出行数 | **塌缩** → 每组 1 行 | **不塌缩** → 每行一行 |
| 看到的"邻居" | 同组所有行 → 算一个总值 | 同窗口所有行 → **每行算一个值** |
| 典型用例 | 每个 country 的总销售额 | 每个 invoice 算占其 country 总额的比例 |

### 一句话直觉 / One-line intuition

> 窗口函数 = "对每一行，看一下它周围的一组邻居，算出一个值"。**原表的每一行都保留**。
> Per-row, peek at a neighbourhood, compute a value. **Every source row survives.**

### 1.2 最简语法 / Minimal syntax

```sql
SELECT
    name,
    price,
    AVG(price) OVER ()       AS overall_avg,        -- ← 窗口函数 / window fn
    price - AVG(price) OVER () AS diff_from_avg
FROM track;
```

`OVER ()` 里**啥都不写** = "整张表是一个大窗口"。下面我们演示。
Empty `OVER ()` = "whole table is one window".


In [ ]:
import duckdb
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

# 重建数据集 / Rebuild dataset
conn = duckdb.connect()
conn.sql("""
CREATE TABLE artist (artist_id INT PRIMARY KEY, name VARCHAR, country VARCHAR);
INSERT INTO artist VALUES
    (1,'The Beatles','UK'), (2,'Pink Floyd','UK'), (3,'Miles Davis','US'),
    (4,'Daft Punk','FR'), (5,'Radiohead','UK'), (6,'Anonymous Artist',NULL);

CREATE TABLE album (album_id INT PRIMARY KEY, title VARCHAR, artist_id INT, year INT);
INSERT INTO album VALUES
    (1,'Abbey Road',1,1969), (2,'The Dark Side of the Moon',2,1973),
    (3,'The Wall',2,1979), (4,'Kind of Blue',3,1959),
    (5,'Discovery',4,2001), (6,'Random Access Memories',4,2013),
    (7,'OK Computer',5,1997), (8,'Demos (unreleased)',6,2024);

CREATE TABLE track (
    track_id INT PRIMARY KEY, name VARCHAR, album_id INT,
    genre VARCHAR, seconds INT, price DECIMAL(4,2)
);
INSERT INTO track VALUES
    (1,'Come Together',1,'Rock',259,0.99), (2,'Something',1,'Rock',182,0.99),
    (3,'Here Comes the Sun',1,'Rock',185,0.99), (4,'Time',2,'Rock',413,1.29),
    (5,'Money',2,'Rock',382,1.29), (6,'Us and Them',2,'Rock',460,1.29),
    (7,'Another Brick in the Wall',3,'Rock',239,1.29),
    (8,'Comfortably Numb',3,'Rock',382,1.29),
    (9,'So What',4,'Jazz',545,1.49), (10,'Freddie Freeloader',4,'Jazz',586,1.49),
    (11,'Blue in Green',4,'Jazz',337,1.49),
    (12,'One More Time',5,'Electronic',320,1.29),
    (13,'Aerodynamic',5,'Electronic',213,1.29),
    (14,'Digital Love',5,'Electronic',301,1.29),
    (15,'Get Lucky',6,'Electronic',369,1.29),
    (16,'Instant Crush',6,'Electronic',337,1.29),
    (17,'Lose Yourself to Dance',6,'Electronic',353,1.29),
    (18,'Paranoid Android',7,'Rock',384,1.29),
    (19,'Karma Police',7,'Rock',261,1.29),
    (20,'No Surprises',7,'Rock',228,1.29),
    (21,'Untitled Demo 1',8,'Rock',180,0.50),
    (22,'Untitled Demo 2',8,'Rock',195,0.50);

CREATE TABLE customer (customer_id INT PRIMARY KEY, name VARCHAR, country VARCHAR, email VARCHAR);
INSERT INTO customer VALUES
    (1,'Alice Chen','US','alice@example.com'), (2,'Bob Smith','UK','BOB@example.com'),
    (3,'Charlie Davis','US','charlie@example.com'),
    (4,'Diana Park','DE','diana@example.com'), (5,'Ethan Miller','US','ethan@example.com'),
    (6,'Fiona Wong','JP',NULL);

CREATE TABLE invoice (
    invoice_id INT PRIMARY KEY, customer_id INT, track_id INT,
    invoice_date DATE, quantity INT
);
INSERT INTO invoice VALUES
    (1,1,1,DATE '2026-01-05',1), (2,1,9,DATE '2026-01-05',2),
    (3,2,4,DATE '2026-01-10',1), (4,2,18,DATE '2026-01-10',1),
    (5,3,5,DATE '2026-02-12',1), (6,3,6,DATE '2026-02-12',1),
    (7,3,7,DATE '2026-02-12',3), (8,4,15,DATE '2026-02-20',1),
    (9,4,12,DATE '2026-02-20',1), (10,4,13,DATE '2026-02-20',1),
    (11,5,9,DATE '2026-03-01',1), (12,5,10,DATE '2026-03-01',1),
    (13,5,11,DATE '2026-03-01',1), (14,5,4,DATE '2026-03-05',2),
    (15,1,18,DATE '2026-03-15',1), (16,1,19,DATE '2026-03-15',1),
    (17,2,15,DATE '2026-04-01',1), (18,3,14,DATE '2026-04-10',2),
    (19,4,8,DATE '2026-05-02',1), (20,5,1,DATE '2026-05-20',1);
""")
print(f"duckdb : {duckdb.__version__}")
print(f"tables : {conn.sql('SHOW TABLES').df()['name'].tolist()}")


In [ ]:
# 最简窗口：整张 track 表算一个总均值挂到每行
# Simplest window: global average attached to every row
conn.sql("""
    SELECT
        name,
        price,
        ROUND(AVG(price) OVER (), 3)              AS overall_avg,
        ROUND(price - AVG(price) OVER (), 3)      AS diff_from_avg
    FROM track
    ORDER BY price DESC
    LIMIT 6;
""").df()


**注意**：22 行原数据全部保留，每行多了一个"全店平均价"和"差额"。这就是窗口函数的精髓。
22 source rows all preserved; each has the global average and a delta. That's the essence.

如果用 `GROUP BY`，你会得到 1 行 1 列；要附加到原表还得 `JOIN` 一次。窗口函数**一步搞定**。
With `GROUP BY` you'd get one row; attaching back requires a JOIN. Windows do it in one step.


<a id="2"></a>
## 2. `OVER` 子句的三大要素 / The Three Parts of `OVER`

```sql
function() OVER (
    PARTITION BY <cols>      ← 分组（像 GROUP BY，但不塌缩）
    ORDER BY    <cols>       ← 窗口内排序（影响累计 / 排名 / LAG / LEAD）
    ROWS BETWEEN ... AND ... ← 帧 (frame)：在窗口里再"切片"
)
```

| 元素 / Part | 作用 / Purpose |
|---|---|
| **PARTITION BY** | 把数据按列分桶；每个桶各自独立计算 |
| **ORDER BY** | 在桶内按某列排序；**有序后才能算累计、排名、LAG/LEAD** |
| **ROWS BETWEEN** | 帧子句；进一步限定"看哪几行邻居"（默认行为容易踩坑，下面讲）|

三者**都可省**，省略时含义：
All three are optional. Default semantics:
- 无 `PARTITION BY` → 整表一个桶 / whole table
- 无 `ORDER BY` → 没有"上一行/下一行"概念，所有行都"等价"
- 无 `ROWS BETWEEN` → ⚠ **默认帧**：`RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`（如果指定了 `ORDER BY`）；否则整个桶


<a id="3"></a>
## 3. 排名函数 / Ranking Functions

四个核心排名函数 + 一个分桶：
Four ranks + one bucketing function:

| 函数 | 同分 5 个第 3 名后下一名是 | 用途 |
|---|---|---|
| `ROW_NUMBER()` | 不会有同分；**强制** 1,2,3,4,5,6... | 严格"挑一个"——top-1 / top-N per group |
| `RANK()` | 1,2,3,3,3,3,3,**8** (跳过 4-7) | 比赛排名（有并列就并）|
| `DENSE_RANK()` | 1,2,3,3,3,3,3,**4** (不跳) | 业务分级（铜银金，不在意基数）|
| `NTILE(N)` | 分成 N 个等大桶，编号 1..N | 把用户分 quintile / decile |


In [ ]:
# 演示 4 种排名 / Demo all four
conn.sql("""
    SELECT
        name,
        genre,
        seconds,
        ROW_NUMBER() OVER (PARTITION BY genre ORDER BY seconds DESC) AS rn,
        RANK()       OVER (PARTITION BY genre ORDER BY seconds DESC) AS rk,
        DENSE_RANK() OVER (PARTITION BY genre ORDER BY seconds DESC) AS dr,
        NTILE(4)     OVER (PARTITION BY genre ORDER BY seconds DESC) AS quartile
    FROM track
    ORDER BY genre, rn;
""").df()


**重点观察**：
- 同 genre 里**按 `seconds DESC` 排序**，每个 genre **独立排名**（PARTITION BY genre 的作用）
- `ROW_NUMBER` 没有重复——**总是** 1, 2, 3, ...
- 如果两首歌时长相同，`RANK` 会给同名次但跳号；`DENSE_RANK` 给同名次不跳号
- `NTILE(4)` 把每个 genre 分成 4 个等大桶

### 🎯 Top-N per group: 最经典的窗口函数面试题

```sql
WITH ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY group_col ORDER BY metric DESC) AS rn
    FROM ...
)
SELECT * FROM ranked WHERE rn <= 3;
```

**记住这个模板** —— 90% 的"每个 X 的 top-N" 都用它。
**Memorize this pattern** — 90% of "top-N per X" problems.


In [ ]:
# 每个 genre 时长最长的 3 首歌 / Top-3 longest tracks per genre
conn.sql("""
    WITH ranked AS (
        SELECT
            name, genre, seconds,
            ROW_NUMBER() OVER (PARTITION BY genre ORDER BY seconds DESC) AS rn
        FROM track
    )
    SELECT name, genre, seconds
    FROM ranked
    WHERE rn <= 3
    ORDER BY genre, seconds DESC;
""").df()


**这一段 SQL** 就替代了 1.4 节那个又慢又难懂的 correlated subquery。窗口函数：1，相关子查询：0。
This window-function version replaces the slow correlated subquery from 1.4. Window: 1, correlated: 0.

### 💡 `ROW_NUMBER` vs `RANK` vs `DENSE_RANK` 选哪个

| 需求 / Need | 用 |
|---|---|
| "每组**严格**选 1 个 / 选前 N 个" | `ROW_NUMBER` |
| "比赛排名（并列就跳号）" | `RANK` |
| "用户等级（铜银金）" | `DENSE_RANK` |


<a id="4"></a>
## 4. 偏移函数 / Offset Functions

让你**取上一行 / 下一行 / 第一行 / 最后一行**——做"环比 / 同期对比 / 看趋势"必备。
Get prev / next / first / last row in the window — essential for period-over-period and trend analysis.

| 函数 | 含义 |
|---|---|
| `LAG(col, n=1, default)` | 当前行**前** $n$ 行的 `col` 值（不够时返回 `default`）|
| `LEAD(col, n=1, default)` | 当前行**后** $n$ 行的 `col` 值 |
| `FIRST_VALUE(col)` | 窗口内**第一行**的 `col` |
| `LAST_VALUE(col)` | 窗口内**最后一行**的 `col`（⚠ 有帧坑！见下）|


In [ ]:
# 每个客户连续的购买 + 与上次购买间隔的天数
# Per-customer purchase sequence + days since last purchase
conn.sql("""
    SELECT
        c.name                                                       AS customer,
        i.invoice_date,
        LAG(i.invoice_date) OVER (
            PARTITION BY i.customer_id ORDER BY i.invoice_date
        ) AS prev_purchase,
        i.invoice_date - LAG(i.invoice_date) OVER (
            PARTITION BY i.customer_id ORDER BY i.invoice_date
        ) AS days_since_prev
    FROM invoice  AS i
    JOIN customer AS c USING (customer_id)
    ORDER BY c.name, i.invoice_date;
""").df()


**看 Alice**：
- 第 1 次买（2026-01-05）→ `prev_purchase = NULL`（窗口里没有"前面"）
- 第 2 次（2026-03-15）→ `prev_purchase = 2026-01-05`，间隔 69 天
- For Alice, 1st purchase → prev=NULL; 2nd → prev=Jan 5, gap=69 days.

### LAG/LEAD 的典型用法 / Typical use cases

- **同比/环比**：`metric - LAG(metric)` 算 delta
- **回购分析**：相邻两次购买的间隔
- **会话化**：相邻两次 event 的间隔 > 30 min 就开新 session
- **趋势线**：`metric / LAG(metric) - 1` 算变化率


<a id="5"></a>
## 5. 聚合作为窗口 / Aggregates as Windows

任何聚合函数（`SUM`/`AVG`/`COUNT`/`MIN`/`MAX`...）后面跟 `OVER(...)` 就变成窗口聚合。
Any aggregate followed by `OVER(...)` becomes a windowed aggregate.

```sql
SUM(amount) OVER (PARTITION BY user_id)              -- 每个用户的总和（不塌缩）
AVG(price) OVER (PARTITION BY genre)                 -- 每个 genre 的均价附加到每行
COUNT(*) OVER (PARTITION BY country)                 -- 每个国家有几行
```


In [ ]:
# 每首歌的"价格 + 同 genre 的均价 + 自己占 genre 总价比"
# Each track: price + genre mean + share of genre total
conn.sql("""
    SELECT
        name,
        genre,
        price,
        ROUND(AVG(price) OVER (PARTITION BY genre), 3)              AS genre_avg_price,
        ROUND(price - AVG(price) OVER (PARTITION BY genre), 3)      AS diff_from_genre_avg,
        ROUND(100.0 * price / SUM(price) OVER (PARTITION BY genre), 2) AS pct_of_genre_total
    FROM track
    ORDER BY genre, price DESC
    LIMIT 10;
""").df()


**这种"附加聚合到每行"的需求**用 GROUP BY 要先聚合再 JOIN 回去。窗口函数**一句搞定**——核心威力。
The "attach group aggregate to each row" pattern needs GROUP-BY-then-JOIN otherwise — windows do it in one statement.


<a id="6"></a>
## 6. ⭐ 帧子句 / The Frame Clause

```sql
function() OVER (
    PARTITION BY ...
    ORDER BY    ...
    ROWS BETWEEN <start> AND <end>
)
```

帧子句决定"窗口内**到底看哪几行**"——这是窗口函数最绕但最强大的部分。
The frame clause decides exactly which rows the function sees — the trickiest but most powerful part.

### 6.1 边界关键字 / Frame bounds

| 关键字 / Keyword | 含义 / Meaning |
|---|---|
| `UNBOUNDED PRECEDING` | 桶的第一行 |
| `n PRECEDING` | 当前行**前** $n$ 行 |
| `CURRENT ROW` | 当前行 |
| `n FOLLOWING` | 当前行**后** $n$ 行 |
| `UNBOUNDED FOLLOWING` | 桶的最后一行 |

### 6.2 三种最常用的帧 / Three canonical frames

```sql
-- 累计 (running total): 从开头到当前
ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW

-- 滑动窗口 (rolling): 当前行 + 前后各 N 行
ROWS BETWEEN 3 PRECEDING AND 3 FOLLOWING

-- 整个桶 (whole partition)
ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
```

### 6.3 ⚠ ROWS vs RANGE

```sql
ROWS BETWEEN <n> PRECEDING AND CURRENT ROW   -- 看**物理行数**
RANGE BETWEEN <n> PRECEDING AND CURRENT ROW  -- 看**ORDER BY 值的差**
```

`ROWS` 看**几行**；`RANGE` 看**值的差**。99% 场景用 `ROWS`，`RANGE` 处理日期/数值范围才有用。
ROWS counts physical rows; RANGE counts ORDER BY value distance. Mostly use ROWS.

### 6.4 ⚠ 默认帧的"坑" / Default frame pitfall

如果你写了 `ORDER BY` 但**没写** `ROWS BETWEEN`，**默认帧是**：
```
RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
```

这意味着**默认就是累计**——不是整桶！很多人在这里翻车。
The default with ORDER BY is "running" — not "whole partition"! A common bug source.

```sql
-- 这两个 NOT 等价 / These are NOT equivalent
SUM(x) OVER (PARTITION BY g)              -- 整组总和 / total
SUM(x) OVER (PARTITION BY g ORDER BY ts)  -- 累计 / running (默认帧)
```


In [ ]:
# 演示默认帧的"坑" / The default-frame trap
# 同一个表达式只差一个 ORDER BY，结果完全不同
print("--- 不加 ORDER BY: 整组总和 (no ORDER BY: total per group) ---")
print(conn.sql("""
    SELECT
        name, genre,
        SUM(seconds) OVER (PARTITION BY genre) AS sum_no_order
    FROM track
    WHERE genre = 'Rock'
    ORDER BY name
    LIMIT 5;
""").df())

print("\n--- 加 ORDER BY: 累计 (with ORDER BY: running by default) ---")
print(conn.sql("""
    SELECT
        name, genre, seconds,
        SUM(seconds) OVER (PARTITION BY genre ORDER BY name) AS sum_with_order
    FROM track
    WHERE genre = 'Rock'
    ORDER BY name
    LIMIT 5;
""").df())


**关键差别**：上半表 `sum_no_order` 是 Rock 的总秒数（5234），整组相同；下半表 `sum_with_order` 是按 name 排序的**累计和**。
The "no ORDER BY" version repeats the same total on every row; the "with ORDER BY" version accumulates.

**经验法则 / Rule**：
- 想要**整组聚合** → 不加 ORDER BY
- 想要**累计** → 加 ORDER BY + 不需要显式 frame
- 想要**自定义帧（滑动/前后 N 行）** → 必须**同时**加 ORDER BY 和 ROWS BETWEEN


<a id="7"></a>
## 7. 累计聚合 / Running Totals


In [ ]:
# 每个客户的累计消费 / Running total of spend per customer
conn.sql("""
    SELECT
        c.name                                                          AS customer,
        i.invoice_date,
        ROUND(i.quantity * t.price, 2)                                  AS this_order,
        ROUND(
            SUM(i.quantity * t.price) OVER (
                PARTITION BY i.customer_id
                ORDER BY i.invoice_date, i.invoice_id
            ),
            2
        ) AS running_total
    FROM invoice  AS i
    JOIN customer AS c USING (customer_id)
    JOIN track    AS t USING (track_id)
    ORDER BY c.name, i.invoice_date, i.invoice_id;
""").df()


**Alice 的累计** 一行行往上加：0.99 → 3.97 → 5.26 → 6.55。这是真实工作里**最常用的窗口函数模式**。
Alice's running total accumulates monotonically — the most-used windowed pattern in real work.


<a id="8"></a>
## 8. 滑动窗口 / Rolling Windows + 7-day MA

```sql
AVG(metric) OVER (
    ORDER BY ts
    ROWS BETWEEN 6 PRECEDING AND CURRENT ROW    -- 7 行的滑动均线 / 7-row MA
)
```

注意：**`ROWS BETWEEN 6 PRECEDING AND CURRENT ROW`** 给的是**最近 7 行**（包括当前行），不是 "前 6 行"。
Frame counts the current row too — 6 preceding + current = 7 rows.


In [ ]:
# 先生成"每天销售额"序列 / Build a daily-revenue series
conn.sql("""
    CREATE OR REPLACE VIEW daily AS
    SELECT
        invoice_date                            AS day,
        SUM(i.quantity * t.price)               AS revenue
    FROM invoice AS i JOIN track AS t USING (track_id)
    GROUP BY invoice_date
    ORDER BY invoice_date;
""")
print(conn.sql("SELECT * FROM daily ORDER BY day").df())


In [ ]:
# 累计销售 + 3 日滑动均线 / Running total + 3-day moving average
# 数据稀疏，所以用 3 日窗口而不是 7 日 / Use 3-day window since data is sparse
conn.sql("""
    SELECT
        day,
        revenue,
        ROUND(SUM(revenue) OVER (ORDER BY day), 2)                    AS running_total,
        ROUND(AVG(revenue) OVER (
            ORDER BY day
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ), 2) AS ma3
    FROM daily
    ORDER BY day;
""").df()


**3 日 MA**：每行是"当天 + 前 2 天"3 天的平均。第一行只有自己 → 等于 revenue；第二行是自己和前 1 天的平均，以此类推。
3-day MA: each row is the average of itself + the previous 2 days. First row averages just itself; second row averages 2; from row 3 onward, full 3-row average.

> 💡 **股票界、监控仪表板里到处都是 MA-N / EWMA**。
> Moving averages and EWMA are everywhere in finance and monitoring.

### 滑动窗口的另一个变种：累积去重计数

```sql
COUNT(DISTINCT user_id) OVER (
    ORDER BY day
    RANGE BETWEEN INTERVAL '7' DAY PRECEDING AND CURRENT ROW
)
-- "过去 7 天活跃过的 unique users" (WAU)
```

注意这里用 `RANGE` 不是 `ROWS`——因为日期不连续，要按"日期距离"切窗口。
Use `RANGE` for date-based windows; `ROWS` for row-count windows.


<a id="9"></a>
## 9. `PERCENT_RANK` & `CUME_DIST`

把排名归一化到 $[0, 1]$ —— 适合**分位数报告**。
Normalize ranks to $[0, 1]$ — handy for percentile reports.

| 函数 | 公式 |
|---|---|
| `PERCENT_RANK()` | $(rank - 1) / (n - 1)$ |
| `CUME_DIST()` | (≤ 当前值的行数) / $n$ |


In [ ]:
# 每首歌在 genre 内的"价格分位" / Price percentile within genre
conn.sql("""
    SELECT
        name, genre, price,
        ROUND(PERCENT_RANK() OVER (PARTITION BY genre ORDER BY price), 3) AS pct_rank,
        ROUND(CUME_DIST()    OVER (PARTITION BY genre ORDER BY price), 3) AS cume_dist
    FROM track
    ORDER BY genre, price;
""").df()


<a id="10"></a>
## 10. ⭐ 实战 / Hands-on

把窗口函数甩在三大经典面试题上。
Three classic interview problems.


In [ ]:
# Q1: 每个 genre 时长最长的 1 首歌 (top-1 per group)
# Top-1 per group: longest track per genre
conn.sql("""
    WITH ranked AS (
        SELECT
            name, genre, seconds,
            ROW_NUMBER() OVER (PARTITION BY genre ORDER BY seconds DESC) AS rn
        FROM track
    )
    SELECT name AS longest_track, genre, seconds
    FROM ranked
    WHERE rn = 1;
""").df()


In [ ]:
# Q2: 每个客户的"首次 / 末次购买日期"
# First and last purchase date per customer
conn.sql("""
    SELECT DISTINCT
        c.name,
        FIRST_VALUE(i.invoice_date) OVER (
            PARTITION BY i.customer_id
            ORDER BY i.invoice_date
        ) AS first_purchase,
        LAST_VALUE(i.invoice_date) OVER (
            PARTITION BY i.customer_id
            ORDER BY i.invoice_date
            ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING  -- ⚠ 必须明示帧
        ) AS last_purchase
    FROM invoice AS i
    JOIN customer AS c USING (customer_id)
    ORDER BY c.name;
""").df()


⚠ **`LAST_VALUE` 的著名陷阱**：默认帧是 `UNBOUNDED PRECEDING AND CURRENT ROW`，所以**默认拿到的是当前行的 invoice_date**——不是真的最后！必须**显式写 `UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING`** 才得到整组的最后一行。
**LAST_VALUE gotcha**: default frame only goes up to CURRENT ROW, so you'd get the current row, not the actual last. Must explicitly write `UNBOUNDED FOLLOWING`.

**面试问到这个就赢了一半。**
Knowing this gotcha alone earns interview points.


In [ ]:
# Q3: 同期对比 - 月环比 / Period-over-period: month-over-month
conn.sql("""
    WITH monthly AS (
        SELECT
            DATE_TRUNC('month', invoice_date)                AS month,
            SUM(i.quantity * t.price)                        AS revenue
        FROM invoice AS i JOIN track AS t USING (track_id)
        GROUP BY 1
    )
    SELECT
        month,
        ROUND(revenue, 2)                                            AS revenue,
        ROUND(LAG(revenue) OVER (ORDER BY month), 2)                 AS prev_month_revenue,
        ROUND(revenue - LAG(revenue) OVER (ORDER BY month), 2)       AS delta,
        ROUND(
            100.0 * (revenue - LAG(revenue) OVER (ORDER BY month))
            / NULLIF(LAG(revenue) OVER (ORDER BY month), 0),
            1
        ) AS pct_change
    FROM monthly
    ORDER BY month;
""").df()


**注意 `NULLIF(divisor, 0)`** —— 1.3 节学过的防 0 除技巧，配合 LAG 必用（第一行的 LAG 是 NULL，NULL ≠ 0 但保险起见还是写）。
Always pair LAG with NULLIF for safety.


In [ ]:
# Q4 (BONUS): 连续 N 天 — gaps-and-islands
# "找连续 2 天都有销售的日期"
# Classic "gaps and islands": find days that are part of a 2+ day streak

# 思路：每天加上一个"等差列"（row_num），如果"日期 - row_num"相同就说明连续
# Idea: subtract row_number from date — same difference = same island
conn.sql("""
    WITH daily AS (
        SELECT DISTINCT invoice_date AS day
        FROM invoice
    ),
    indexed AS (
        SELECT
            day,
            ROW_NUMBER() OVER (ORDER BY day)                AS rn,
            day - ROW_NUMBER() OVER (ORDER BY day) * INTERVAL 1 DAY AS island_key
        FROM daily
    ),
    islands AS (
        SELECT
            island_key,
            MIN(day) AS start_day,
            MAX(day) AS end_day,
            COUNT(*) AS streak_length
        FROM indexed
        GROUP BY island_key
    )
    SELECT start_day, end_day, streak_length
    FROM islands
    WHERE streak_length >= 1     -- 这个 toy 数据集稀疏，演示 >= 1 / sparse data
    ORDER BY start_day;
""").df()


**"gaps-and-islands" 解法 = `日期 - row_number = 常数 ⇒ 同 island`**。
The classic gaps-and-islands trick: `date - row_number = constant ⇒ same streak`.

数据少，演示效果有限；真实大数据里**全部连续登录 7 天的用户**都用这个思路。
With more data you'd see clear streaks; this is how "consecutive 7-day login" queries work.


<a id="11"></a>
## 11. 小结 / Summary

### 概念地图 / Concept map

```
窗口函数 = 不塌缩的聚合
  │
  OVER (
      PARTITION BY <cols>   ← 像 GROUP BY 但不塌缩
      ORDER BY    <cols>    ← 桶内排序（开启累计/排名/LAG/LEAD）
      ROWS BETWEEN          ← 帧子句：定义"看哪几行"
  )
  │
  ├── 排名函数
  │     ├── ROW_NUMBER  ⭐  严格 1,2,3,...
  │     ├── RANK         比赛式（跳号）
  │     ├── DENSE_RANK   等级式（不跳）
  │     └── NTILE(N)     等大分桶
  │
  ├── 偏移函数
  │     ├── LAG / LEAD ⭐    上/下 N 行 → 环比、间隔
  │     ├── FIRST_VALUE       窗口第一行
  │     └── LAST_VALUE  ⚠     默认帧坑！必须 UNBOUNDED FOLLOWING
  │
  ├── 聚合窗口
  │     ├── SUM / AVG / COUNT / MIN / MAX OVER (...)
  │     └── 加 PARTITION BY 算"每组指标附到每行"
  │
  └── 帧 + 时间序列
        ├── 累计 ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ├── 滑动 ROWS BETWEEN N PRECEDING AND CURRENT ROW    （N 行均线）
        └── RANGE BETWEEN INTERVAL '7' DAY PRECEDING …      （时间范围 WAU 等）
```

### 💡 必背模板 / Must-memorize templates

```sql
-- Top-N per group ⭐
WITH ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY g ORDER BY metric DESC) AS rn
    FROM t
)
SELECT * FROM ranked WHERE rn <= N;

-- 环比 / Period-over-period
LAG(metric) OVER (PARTITION BY user_id ORDER BY ts)

-- 累计 / Running total
SUM(amount) OVER (PARTITION BY user_id ORDER BY ts)

-- N-row 滑动平均 / N-row moving average
AVG(metric) OVER (
    ORDER BY ts
    ROWS BETWEEN (N-1) PRECEDING AND CURRENT ROW
)

-- 整组聚合附到每行 / Group aggregate on each row
SUM(x) OVER (PARTITION BY g)        -- ⚠ 不加 ORDER BY 才是整组！
```

### 💡 工业速查 / Industry cheat sheet

| 场景 / Task | 窗口函数 |
|---|---|
| Top-1 / Top-N per group | `ROW_NUMBER() OVER (PARTITION BY g ORDER BY x DESC)` |
| 环比 / MoM / YoY | `LAG(x) OVER (ORDER BY ts)` |
| 累计 GMV | `SUM(amt) OVER (ORDER BY day)` |
| 7 日 MA | `AVG(x) OVER (ORDER BY day ROWS BETWEEN 6 PRECEDING AND CURRENT ROW)` |
| WAU (7 日去重活跃)  | `COUNT(DISTINCT user) OVER (ORDER BY day RANGE BETWEEN INTERVAL '6' DAY PRECEDING AND CURRENT ROW)` |
| 首次 / 末次值 | `FIRST_VALUE(x)` / `LAST_VALUE(x) ⚠UNBOUNDED FOLLOWING⚠` |
| 用户分 quintile | `NTILE(5) OVER (ORDER BY total_spend)` |
| 连续 N 天 | gaps-and-islands: `ts - ROW_NUMBER()` |

### 💡 面试必答 / Interview must-knows

1. **窗口函数 vs GROUP BY**：前者不塌缩行
2. **ROW_NUMBER vs RANK vs DENSE_RANK**：能秒答（同分时谁跳号谁不跳）
3. **Top-N per group 模板** —— 闭眼能写
4. **`LAST_VALUE` 默认帧坑** —— 答得出就赢
5. **默认帧的"坑"**：有 ORDER BY 时默认是累计，不是整组
6. **gaps-and-islands** —— "连续 N 天" 经典套路

### 下一节预告 / Next up

**Part 1.6 · 高级 SQL 题型 / LeetCode-style** —— 用本节学的窗口函数刷一遍**面试高频 SQL 题**：留存率、漏斗、pivot 行转列、sessionization、报表生成。
**Part 1.6 · Advanced SQL Patterns** — drill the high-frequency interview problems with windows: retention, funnels, pivot, sessionization.
